In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.proxy_target_variable import calculate_rfm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [55]:
dff = pd.read_pickle("../data/processed/processed_data.pkl")
df=dff.copy()

In [56]:
df

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult,TransactionHour,TransactionDay,TransactionMonth,TransactionYear,TransactionDayOfWeek
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,...,1000.0,1000,2018-11-15 02:18:49+00:00,2,0,2,15,11,2018,3
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-20.0,20,2018-11-15 02:19:08+00:00,2,0,2,15,11,2018,3
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,...,500.0,500,2018-11-15 02:44:21+00:00,2,0,2,15,11,2018,3
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,...,20000.0,21800,2018-11-15 03:32:55+00:00,2,0,3,15,11,2018,3
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-644.0,644,2018-11-15 03:34:21+00:00,2,0,3,15,11,2018,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95657,TransactionId_89881,BatchId_96668,AccountId_4841,SubscriptionId_3829,CustomerId_3078,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-1000.0,1000,2019-02-13 09:54:09+00:00,2,0,9,13,2,2019,2
95658,TransactionId_91597,BatchId_3503,AccountId_3439,SubscriptionId_2643,CustomerId_3874,UGX,256,ProviderId_6,ProductId_10,airtime,...,1000.0,1000,2019-02-13 09:54:25+00:00,2,0,9,13,2,2019,2
95659,TransactionId_82501,BatchId_118602,AccountId_4841,SubscriptionId_3829,CustomerId_3874,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-20.0,20,2019-02-13 09:54:35+00:00,2,0,9,13,2,2019,2
95660,TransactionId_136354,BatchId_70924,AccountId_1346,SubscriptionId_652,CustomerId_1709,UGX,256,ProviderId_6,ProductId_19,tv,...,3000.0,3000,2019-02-13 10:01:10+00:00,2,0,10,13,2,2019,2


In [57]:
snapshot_date = (
    df["TransactionStartTime"].max()
    + pd.Timedelta(days=1)
)

# Proxy Target Variable
## Create RFM Metrics

In [58]:
rfm_df = calculate_rfm(df)

rfm_df.head()

,CustomerId,Recency,Frequency,Monetary
0,CustomerId_1,84,1,10000
1,CustomerId_10,84,1,10000
2,CustomerId_1001,90,5,30400
3,CustomerId_1002,26,11,4775
4,CustomerId_1003,12,6,32000


In [59]:
print(rfm_df.describe())

           Recency    Frequency      Monetary
count  3742.000000  3742.000000  3.742000e+03
mean     31.461251    25.564404  2.531025e+05
std      27.118932    96.929602  2.715877e+06
min       1.000000     1.000000  5.000000e+01
25%       6.000000     2.000000  6.500000e+03
50%      25.000000     7.000000  3.200000e+04
75%      54.000000    20.000000  1.020600e+05
max      91.000000  4091.000000  1.049000e+08


## Scaling the RFM Features

The **Recency, Frequency, and Monetary (RFM)** variables are measured on different scales:

- **Recency** → Number of days since the customer's last transaction
- **Frequency** → Total number of transactions made by the customer
- **Monetary** → Total value of customer transactions

Because these variables have substantially different ranges, clustering algorithms such as **K-Means** may be disproportionately influenced by features with larger numerical values, particularly the Monetary variable. To prevent this and ensure that each RFM metric contributes equally to the clustering process, the features were standardized using **StandardScaler**, which transforms the data to have a mean of 0 and a standard deviation of 1.

In [60]:
rfm_features = rfm_df[
    ["Recency", "Frequency", "Monetary"]
]

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm_features
)

## Apply K-Means Clustering

In [61]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

rfm_df["Cluster"] = kmeans.fit_predict(
    rfm_scaled
)

## High-Risk Cluster Identification

In [62]:
cluster_summary = (
    rfm_df
    .groupby("Cluster")
    [["Recency", "Frequency", "Monetary"]]
    .mean()
)

print(cluster_summary)

           Recency    Frequency      Monetary
Cluster                                      
0        61.877279     7.720196  8.973793e+04
1        12.715398    34.703720  2.247565e+05
2        23.250000  1104.500000  7.487659e+07


In [63]:
print(cluster_summary)

           Recency    Frequency      Monetary
Cluster                                      
0        61.877279     7.720196  8.973793e+04
1        12.715398    34.703720  2.247565e+05
2        23.250000  1104.500000  7.487659e+07


In [64]:
high_risk_cluster = (
    cluster_summary
    .sort_values(
        by=["Recency", "Frequency", "Monetary"],
        ascending=[False, True, True]
    )
    .index[0]
)

rfm_df["is_high_risk"] = (
    rfm_df["Cluster"] == high_risk_cluster
).astype(int)

In [65]:
rfm_df.head()

,CustomerId,Recency,Frequency,Monetary,Cluster,is_high_risk
0,CustomerId_1,84,1,10000,0,1
1,CustomerId_10,84,1,10000,0,1
2,CustomerId_1001,90,5,30400,0,1
3,CustomerId_1002,26,11,4775,1,0
4,CustomerId_1003,12,6,32000,1,0


## Defining the High-Risk Proxy Target Variable

After segmenting customers using K-Means clustering, the average Recency, Frequency, and Monetary (RFM) values of each cluster were analyzed to identify the least engaged customer group.

The cluster characterized by:

- High Recency (customers had not transacted recently),
- Low Frequency (few transactions), and
- Low Monetary Value (low spending activity),

was identified as the most disengaged customer segment and selected as the **high-risk cluster**.

A binary target variable named **`is_high_risk`** was then created. Customers belonging to the high-risk cluster were assigned a value of **1**, while customers in all other clusters were assigned a value of **0**.

This proxy target variable serves as a substitute for a default label and provides the target required for training and evaluating the credit risk model.

In [68]:
dff = pd.read_pickle("../data/processed/featured.pkl")
customer_features=dff.copy()

In [70]:
target_df = rfm_df[
    ["CustomerId", "is_high_risk"]
]

# Merge target with customer features
model_df = customer_features.merge(
    target_df,
    on="CustomerId",
    how="left"
)



In [71]:
model_df.to_pickle("../data/processed/model_df.pkl")